In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, DoubleType


In [0]:
df = spark.table("workspace.bronze.payments")



In [0]:
# Remove rows with no associated order
df = df.filter(F.col("order_id").isNotNull())

# Only trim columns relevant for joins or filtering 
cols_to_trim = ["order_id", "payment_type"]
for c in cols_to_trim:
    df = df.withColumn(c, F.trim(F.col(c)))

In [0]:
df = df.withColumnRenamed("payment_sequential", "payment_sequence")

In [0]:
df = (
    df
    # Standardizing payment_type
    .withColumn("payment_type", 
                F.lower(F.regexp_replace(F.col("payment_type"), "_", " ")))
    
    # Casting columns
    .withColumn("payment_sequence", 
                F.coalesce(F.col("payment_sequence").cast(IntegerType()), F.lit(1)))
    .withColumn("payment_installments", 
                F.coalesce(F.col("payment_installments").cast(IntegerType()), F.lit(1)))
    .withColumn("payment_value", 
                F.round(F.coalesce(F.col("payment_value").cast(DoubleType()), F.lit(0.0)), 2))
)

In [0]:
df = df.dropDuplicates(["order_id", "payment_sequence", "payment_type"])


In [0]:
df = df.withColumn("silver_processed_time", F.current_timestamp())

In [0]:
df.limit(5).display()

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.order_payments")